# 🚚 Delivery Performance Analysis — SwiftEats

SLA compliance, peak congestion, partner rankings, and zone health.

## Setup

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER','food_user')}:{os.getenv('DB_PASSWORD','')}"
    f"@{os.getenv('DB_HOST','localhost')}:{os.getenv('DB_PORT','5432')}"
    f"/{os.getenv('DB_NAME','food_delivery')}"
)
print("Connected ✓")

## 1. Delivery Time Distribution

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT actual_time_mins FROM deliveries
        WHERE delivery_status = 'Delivered' AND actual_time_mins IS NOT NULL
        AND actual_time_mins < 120
    '''), conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(df['actual_time_mins'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(df['actual_time_mins'].median(), color='orange', linestyle='--', label=f"Median: {df['actual_time_mins'].median():.0f}min")
ax1.axvline(45, color='red', linestyle='--', label='SLA Limit: 45min')
ax1.set_title('Delivery Time Distribution')
ax1.set_xlabel('Minutes')
ax1.legend()

pcts = [10,25,50,75,90,95,99]
vals = [df['actual_time_mins'].quantile(p/100) for p in pcts]
ax2.plot([f'P{p}' for p in pcts], vals, 'o-', color='steelblue', linewidth=2)
ax2.axhline(45, color='red', linestyle='--', label='SLA=45min')
ax2.set_title('Delivery Time Percentiles')
ax2.set_ylabel('Minutes')
ax2.legend()
plt.tight_layout()
plt.show()
print(f"Median: {df['actual_time_mins'].median():.1f}min | P90: {df['actual_time_mins'].quantile(0.9):.1f}min")
print(f"SLA Breach rate: {(df['actual_time_mins']>45).mean()*100:.1f}%")

## 2. Zone SLA Performance

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT z.zone_name, ci.city_name,
               COUNT(*) AS deliveries,
               ROUND(AVG(d.actual_time_mins)::numeric,1) AS avg_time,
               ROUND(COUNT(*) FILTER(WHERE d.sla_breached)*100.0/NULLIF(COUNT(*),0)::numeric,1) AS sla_pct
        FROM deliveries d
        JOIN orders o ON d.order_id = o.order_id
        JOIN zones z ON o.zone_id = z.zone_id
        JOIN cities ci ON z.city_id = ci.city_id
        WHERE d.delivery_status = 'Delivered'
        GROUP BY z.zone_name, ci.city_name
        HAVING COUNT(*) > 50
        ORDER BY sla_pct DESC LIMIT 20
    '''), conn)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#E74C3C' if p > 15 else '#F39C12' if p > 8 else '#27AE60' for p in df['sla_pct']]
bars = ax.barh(df['zone_name'] + ' (' + df['city_name'] + ')', df['sla_pct'], color=colors)
ax.axvline(10, color='orange', linestyle='--', label='Warning: 10%')
ax.axvline(15, color='red', linestyle='--', label='Critical: 15%')
ax.set_title('SLA Breach Rate by Zone (Top 20 worst)')
ax.set_xlabel('SLA Breach %')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Partner Performance Quartiles

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT performance_quartile, vehicle_type,
               COUNT(*) AS partners,
               ROUND(AVG(avg_delivery_time)::numeric,1) AS avg_mins,
               ROUND(AVG(on_time_rate_pct)::numeric,1) AS avg_on_time_pct,
               ROUND(AVG(total_earnings)::numeric,0) AS avg_earnings
        FROM mv_partner_performance
        GROUP BY performance_quartile, vehicle_type
        ORDER BY performance_quartile DESC, partners DESC
    '''), conn)

pivot = df.pivot_table(index='performance_quartile', columns='vehicle_type',
                       values='avg_on_time_pct', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('On-Time Rate % by Performance Quartile and Vehicle Type')
ax.set_xlabel('Performance Quartile (4=Best)')
ax.set_ylabel('On-Time Rate %')
ax.axhline(85, color='red', linestyle='--', label='Target: 85%')
ax.legend(title='Vehicle Type')
ax.set_xticklabels([f'Q{int(q)}' for q in pivot.index], rotation=0)
plt.tight_layout()
plt.show()